In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

import torch
import torchvision
from torchvision import models, transforms
from torchvision.models import (
    ResNet18_Weights,
    MobileNet_V3_Small_Weights,
    EfficientNet_B0_Weights,
)
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import time
import random
import copy
import os
from pathlib import Path
from sklearn.decomposition import PCA

os.makedirs("outputs", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATA_DIR = Path("/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test")
LABELS   = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

random.seed(42)


def load_images(n_per_class=10):
    """Load n images per class. Returns a list of (PIL.Image, label_string) tuples."""
    image_set = []
    for label in LABELS:
        class_dir = DATA_DIR / label
        paths = random.sample(list(class_dir.glob("*.jpg")), n_per_class)
        for path in paths:
            img = Image.open(path).convert("RGB")
            image_set.append((img, label))
    random.shuffle(image_set)
    return image_set
image_set = load_images(n_per_class=10)
print(f"Total images loaded: {len(image_set)}")

# === Display 2x3 sample grid ===
fig, axes = plt.subplots(2, 3, figsize=(10, 7))

for i, label in enumerate(LABELS):
    # Find first image with this label
    for img, lbl in image_set:
        if lbl == label:
            ax = axes[i // 3, i % 3]
            ax.imshow(img)
            ax.set_title(label)
            ax.axis('off')
            break

plt.tight_layout()
plt.savefig("outputs/dataset_sample.png")
plt.show()
plt.close()

#The imageNet doesnt exactly match with the six scene type, But it is a good starting point. The trained model already has the general visual feauture , edges, texture.
#That works well to scene classification, It may not have the direct ouput of mount , but it still recongize the pattern, which Imagenet label as valley , alp .
#This goes to show that having pretraing model is useful , it takes abit of workload , compare to having to start from scratch, we are able to know right away ,
#the visualiaztion of it that it may not have pijnt directly but the catergosiztion of it , gives the idea or pattern how the model behaves. 


#============================= Task 2 ============================
resnet_weights   = ResNet18_Weights.DEFAULT
resnet           = models.resnet18(weights=resnet_weights).to(device).eval()
resnet_preproc   = resnet_weights.transforms()
imagenet_classes = resnet_weights.meta["categories"]

print(f"ResNet18 parameters: {sum(p.numel() for p in resnet.parameters()):,}")

#general - purpose inference function 

import torch 

def run_inference(model, preproc, img, device, classes, top_k=5):
    #imge to(device) ,Preprocess pipeline 
    x = preproc(img).unsqueeze(0).to(device)

    #interface - not training the model, just passing 
    with torch.no_grad():
        logits = model(x)

    #Logits to probabilities 
    probs = torch.softmax(logits, dim=1)[0]

     #Get the top-k highest probability 
    top_probs, top_indxs = torch.topk(probs, k=top_k)

    #class_name , probability tuples for top-k
    return [
        (classes[idx.item()], prob.item())
        for prob, idx in zip(top_probs, top_indxs)
    ]
  #loop running it on the image set
resnet_results = []
for img, true_label in image_set:
    preds = run_inference(resnet, resnet_preproc, img, device, imagenet_classes)
    resnet_results.append({
        "true_label":   true_label,
        "top1_class":   preds[0][0],
        "top1_prob":    preds[0][1],
        "top5_classes": [p[0] for p in preds],
        "top5_probs":   [p[1] for p in preds],
    })

print(f"Processed {len(resnet_results)} images.")


#compute and use panda 
df = pd.DataFrame(resnet_results)
#Overall mean top-1 probability across all images
overall_mean = df["top1_prob"].mean()
#print result 
print(f"Overall mean top-1_prob: {overall_mean:.4f}")

#Mean top-1 probability broken down by true class (which classes does the model feel most and least confident about?)
by_class = df.groupby("true_label")["top1_prob"].mean().sort_values(ascending=False)
print(f"\nMean top-1 probability by true class :\n{by_class}")

#=================== boxplot ===========
import matplotlib.pyplot as plt
import os 

os.makedirs("output", exist_ok=True)

#Group probabilities by true class
classes_in_order = sorted(df["true_label"].unique())
data = [df[df["true_label"] == c]["top1_prob"].values for c in classes_in_order]

#Create a boxplot showing the distribution of top-1 probabilities across the six classes. 
fig , ax = plt.subplots(figsize=(10, 6))
ax.boxplot(data, labels=classes_in_order)
ax.set_ylabel("Top-1 probability")
ax.set_xlabel("True class")
ax.set_title("ResNet18 top-1 by class")
ax.set_ylim(0,1)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("output/resnet18_confidence_by_class.png", dpi=150)
plt.show()

#comment 
#The hgh confidence and high accuracy ca not be the same because a model can be 99% sure and still be wrong especially if the image is unknown.
#The softmax score shows just how strongly the model voted, its not 100% right.
#top-1 prob >= 0.85 is auto tag the photo ,top-1 prob 0.50 - 0.85 is send to a human reviewer and top-1 prob < 0.50 is not tag, mark as unknown.
#the 0.5 to 0.85 is the middle ground where its not sure enough to trust, but not bad enough to throw away zone and thats where we should look.



In [ ]:
#========== Task 3
#MobileNetV3-Small — designed for mobile and edge deployment
mobile_weights = MobileNet_V3_Small_Weights.DEFAULT
mobilenet      = models.mobilenet_v3_small(weights=mobile_weights).to(device).eval()
mobile_preproc = mobile_weights.transforms()

# EfficientNet-B0 — designed to maximize accuracy per unit of compute
effnet_weights = EfficientNet_B0_Weights.DEFAULT
efficientnet   = models.efficientnet_b0(weights=effnet_weights).to(device).eval()
effnet_preproc = effnet_weights.transforms()

# Print parameter counts for all three
for name, m in [("ResNet18",          resnet),
                ("MobileNetV3-Small", mobilenet),
                ("EfficientNet-B0",   efficientnet)]:
    params = sum(p.numel() for p in m.parameters())
    print(f"{name:22s}  {params:>12,} parameters")


#======= Comment ======
#what does a smaller parameter count imply about a model's capacity? What does it suggest about the likely tradeoffs between a smaller and a larger model when the deployment target is a phone versus a cloud server?
#What the smaller parameter count imply that the model capacity , shows fewer pattern. Now the tradeoff on a phone you pick the smallest model to meet your accuracy bar, while on the cloud server , theres no device limitation and its a bigger model.

In [ ]:
#====== Continue Task 3 
#reuse run_inference exactly like Task 2
#======== mobilenet_results =========
mobilenet_results = []
for img, true_label in image_set:
    preds = run_inference(mobilenet, mobile_preproc, img, device, imagenet_classes)
    mobilenet_results.append({
        "true_label":   true_label,
        "top1_class":   preds[0][0],
        "top1_prob":    preds[0][1],
        "top5_classes": [p[0] for p in preds],
        "top5_probs":   [p[1] for p in preds],
    })

#=========== effnet_results ===========
#store results in effnet_results 
effnet_results = []
for img, true_label in image_set:
    preds = run_inference(efficientnet, effnet_preproc, img, device, imagenet_classes)
    #add to effnet_results
    effnet_results.append({
    "true_label": true_label,
        "top1_class": preds[0][0],
        "top1_prob": preds[0][1],
        "top5_classes": [p[0] for p in preds],
        "top5_probs": [p[1] for p in preds]
})
print(f"MobileNet processed: {len(mobilenet_results)} images")
print(f"EfficientNet processed: {len(effnet_results)} images")

#Pick one image smaple per class 
sample_idxs = {}
for i, (img, label) in enumerate(image_set):
    if label not in sample_idxs:
        sample_idxs[label] = i 
    if len(sample_idxs) == 6:
        break

#Get the top 3 from each model , stored top 5 , the top 3 is just the slice 
def top3(results, i):
    return list(zip(results[i]["top5_classes"][:3], results[i]["top5_probs"][:3]))

#create comparison grid for 6 images 

fig, axes = plt.subplots(6, 2, figsize=(12, 20),
                         gridspec_kw={"width_ratios":[1, 1.4]})

#loop 
for row, label in enumerate(LABELS):
    i = sample_idxs[label]
    img, true_label = image_set[i]

#left column for image 
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"True: {true_label}", fontsize=11)
    axes[row, 0].axis("off")
#right column for texy with each model 
    axes[row, 1].axis("off")
    lines = []
    for model_name, results in [
        ("ResNet18", resnet_results),
        ("MobileNetV3", mobilenet_results),
        ("EfficientNet", effnet_results),
    ]:
        lines.append(f"{model_name}:") #add 
        for cls, prob in top3(results, i):
            lines.append(f" {cls:30s} {prob:.3f}")
            lines.append("")# blank line between models 

    axes[row, 1].text(0, 1, "\n".join(lines),
                     fontsize=9, family="monospace",
                      verticalalignment="top")
#printoutput  
plt.suptitle("Top-3 predictions per model, one image per class", fontsize=14)
plt.tight_layout()
plt.savefig("outputs/model_comparison_grid.png", dpi=150, bbox_inches="tight")
plt.show()
#========= Comment 
#The model does agree with the top 1 mostly. All three picked"valley" for the mountain image, and all three picked "prison" for the
#street image. They line up when the scene has obvious visual cues.

#They disagree on harder images. For the glacier, ResNet said "spaceshuttle," MobileNet said "ski," and EfficientNet said "airliner" 
#none are right, and all three guessed differently. However an average of their ensemble (average probabilities) would help as it would combine them and cancel them out.

#MobileNet carried more noise since its the smallest model , Efficientnet guesses stays in the right category. ResNet does it as well.

In [ ]:
def benchmark_model(model, preprocess, image_set, device, n_warmup=5):
    """
    Benchmark single-image inference speed.
    Returns mean latency in milliseconds per image.
    """
    # Warm up the GPU — the first few calls are slower due to CUDA initialization
    for img, _ in image_set[:n_warmup]:
        tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            _ = model(tensor)

    # Timed run — synchronize before and after to get accurate GPU timing
    torch.cuda.synchronize()
    start = time.time()

    for img, _ in image_set:
        tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            _ = model(tensor)

    torch.cuda.synchronize()
    elapsed = time.time() - start

    return (elapsed / len(image_set)) * 1000  # milliseconds per image

resnet_ms  = benchmark_model(resnet,       resnet_preproc,  image_set, device)
mobile_ms  = benchmark_model(mobilenet,    mobile_preproc,  image_set, device)
effnet_ms  = benchmark_model(efficientnet, effnet_preproc,  image_set, device)

print(f"ResNet18:           {resnet_ms:.2f} ms/image")
print(f"MobileNetV3-Small:  {mobile_ms:.2f} ms/image")
print(f"EfficientNet-B0:    {effnet_ms:.2f} ms/image")


#============= Bar Chart
models_names = ["ResNet18", "MobileNetV3-small", "EfficientNet-BO"]
latencies    = [resnet_ms, mobile_ms, effnet_ms]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models_names, latencies, color=["steelblue", "seagreen", "indianred"])
ax.set_ylabel("Latency (ms/image)")
ax.set_xlabel("Model")
ax.set_title("Inference latency per model (single image)")

#interpret each bar with its value 
#loop
for bar, val in zip(bars, latencies):
    ax.text(bar.get_x() + bar.get_width()/2, val, f"{val:.2f}",
            ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("outputs/inference_speed.png", dpi=150)
plt.show()

#Real-time constraint 

#To hit 50 images per second, each image has to finish in 1000 / 50 = 20 ms
#or less. All three of my models came in under that ResNet18 at 4.44 ms,
#MobileNetV3-Small at 7.82 ms, and EfficientNet-B0 at 11.38 ms so any of
#them can keep up with the 50 fps rate on GPU.

#One thing I noticed is that ResNet18 was the fastest even though it has
#the most parameters. That's because GPUs handle its simple stacked conv
#layers really well, while MobileNet and EfficientNet use fancier ops that
#are smaller but not always faster on a GPU. Those two are built to win on
#phones and CPUs, not big GPUs. On CPU the numbers would be slower and
#probably only MobileNet would stay under 20 ms.

#which model for which deployment:**
#Cloud pipeline :  EfficientNet. Better accuracy than MobileNet for similar latency, and on a server you're optimizing
#cost per image, not battery. The extra accuracy is worth the small compute bump.

#second would be device mobile app: MobileNetV3-Small. It was literally
#designed for phones: smallest parameter count, fastest inference,
#lowest battery drain. A small accuracy hit is acceptable when the
#alternative is a laggy or hot phone.

#Third be QC system : ResNet18 (or even better, the largest
#model available). When mistakes are costly, speed comes second to
#accuracy. You'd also want to combine it with a confidence threshold
#and human review for low-confidence cases. Just like Task 2.

In [ ]:
#task 5 

import copy

feature_extractor = copy.deepcopy(resnet)
feature_extractor.fc = torch.nn.Identity()   # remove the classification head
feature_extractor    = feature_extractor.to(device).eval()

def extract_features(model, preprocess, image, device):
    """Extract a feature vector from an image using the truncated CNN."""
    tensor   = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        features = model(tensor)
    return features.squeeze().cpu().numpy()

# Extract features for all images
feature_vectors = []
true_labels     = []

for img, label in image_set:
    feat = extract_features(feature_extractor, resnet_preproc, img, device)
    feature_vectors.append(feat)
    true_labels.append(label)

feature_matrix = np.array(feature_vectors)
print(f"Feature matrix shape: {feature_matrix.shape}")
# Expected: (60, 512) — 60 images, 512-dimensional feature vector each
pca          = PCA(n_components=2)
features_2d  = pca.fit_transform(feature_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
colors  = plt.cm.tab10(np.linspace(0, 1, len(LABELS)))

for i, label in enumerate(LABELS):
    mask = [l == label for l in true_labels]
    ax.scatter(
        features_2d[mask, 0],
        features_2d[mask, 1],
        label=label, color=colors[i], s=60, alpha=0.75
    )

ax.legend()
ax.set_title("ResNet18 Feature Embeddings (PCA to 2D)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.savefig("outputs/feature_embeddings.png")
plt.show()

#==== Comment 
#Yes the images from  the same class are mostly cluster together in 2D plot. Forest image sit near other forest images, glacier near glacier so on.
#This shows that the ResNet18 already learned useful visual features from ImageNet to edges, colors, textures. The pretrained model can tell aprt these scenes from the embedding of it. 
#Some classes overlap which make sense because they share visual feautures like snow and rocky texture.

# With only 500 X-ray images I would start with **feature extraction**
#(freeze the pretrained layers, train only a new final layer). 500 images
#is too small to safely update millions of weights without overfitting.
#Feature extraction reuses what ResNet18 already learned and only trains
#a small classifier on top. If accuracy isn't good enough I could try
#fine-tuning the last few layers afte


#============== Task 6 =========

* Model Comparison:
* EfficientNet gave the most sensible top-5 predictions for outdoor
  scenes (its top-3 for the sea image was sandbar, seashore, lakeside,all water related).
* MobileNet was the noisiest, like guessing planetarium for the sea image.
* On speed, ResNet18 was fastest at 4.44 ms, MobileNet at 7.82 ms, and EfficientNet at 11.38 ms.
* Overall EfficientNet had the best prediction quality, but ResNet18 was the best mix of decent predictions and fastest speed on GPU.(It didnt have my laptop lagging)

**Confidence Calibration**
* From the Task 2 boxplot, ResNet18 was most confident on forest and glacier images, and least confident on buildings and street.
* That makes sense because forest and glacier have very distinct visual features (lots of green, lots of snow and ice), while buildings andstreet look kinda similar and overlap a lot (both have walls, windows,pavement).

**Production Recommendation**
For classifying user uploaded outdoor photos into the six scene categories,I would start with EfficientNet since it gave the most sensible predictions and still runs fast enough for a pipeline. Thepreprocessing would need to match what the model expects: resize, center
crop, convert to tensor, and normalize using the weights.transforms() pipeline so the inputs are consistent. One risk I would flag is that ImageNet doesn't have exact matches for the six Intel classes (no "buildings" label, etc), so we would still need to either fine tune the
final layer on Intel data or map ImageNet outputs to our six classesmanually, otherwise the model will keep guessing close but wrong labels.